# Capítulo 2 – Preparación y Limpieza de Datos
## Sistema de Bicicletas Compartidas – Dataset `hour.csv`

En este capítulo preparamos el conjunto de datos para su uso en los modelos de regresión lineal múltiple. Partimos del archivo original `hour.csv` y avanzamos hasta construir un **dataset limpio y listo para modelar**, con variables numéricas y dummies para las categóricas.


## 1. Carga de librerías y datos

Cargamos nuevamente el archivo `hour.csv` desde la carpeta `data/` del proyecto. Este notebook es autosuficiente (no depende del estado del Capítulo 1 en memoria).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11


In [ ]:
# Ruta relativa al archivo de datos desde la carpeta 'book'
data_path = "../data/hour.csv"

df = pd.read_csv(data_path)
df.head()

### 1.1. Estructura general y valores faltantes

Revisamos el tamaño del dataset, los tipos de datos y la presencia de valores faltantes.


In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

### 1.2. Búsqueda de filas duplicadas

Verificamos si existen filas completamente duplicadas en el dataset.


In [ ]:
n_duplicados = df.duplicated().sum()
n_duplicados

In [ ]:
if n_duplicados > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Se eliminaron {n_duplicados} filas duplicadas.")
else:
    print("No se encontraron filas duplicadas.")

## 2. Tipos de variables y codificación categórica

En el Capítulo 1 identificamos varias variables que representan categorías mediante códigos numéricos. Aquí formalizamos su conversión al tipo `category` y luego creamos variables indicadoras (dummies) para utilizarlas en los modelos.


In [ ]:
categorical_cols = [
    "season", "yr", "mnth", "hr", "holiday",
    "weekday", "workingday", "weathersit"
]

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")

df[categorical_cols].head()

## 3. Selección de variables para el modelo base

Nuestro objetivo será modelar la variable objetivo `cnt` (demanda horaria total).
A partir del análisis exploratorio previo y de la documentación del dataset, definimos un conjunto inicial de variables explicativas:

- Variables ambientales: `temp`, `atemp`, `hum`, `windspeed`.
- Variables temporales: `hr`, `weekday`, `workingday`, `season`, `yr`.
- Condiciones climáticas: `weathersit`.

Las variables `casual` y `registered` también explican `cnt`, pero suelen considerarse más como descomposición de la demanda total que como predictores independientes primarios. En este taller nos centraremos en variables de calendario y clima para el modelo principal.


In [ ]:
target_col = "cnt"
base_features = [
    "temp", "atemp", "hum", "windspeed",
    "hr", "weekday", "workingday", "season", "yr", "weathersit"
]

# Verificamos que todas existen en el DataFrame
missing = [c for c in base_features + [target_col] if c not in df.columns]
missing

In [ ]:
if len(missing) > 0:
    raise ValueError(f"Faltan las siguientes columnas en el dataset: {missing}")
else:
    print("Todas las columnas esperadas están presentes.")

## 4. Ingeniería de características básica

### 4.1. Variable de hora pico (`peak_hour`)

Creamos una variable indicadora que marque las horas típicamente asociadas a picos de uso (por ejemplo, horas laborales de entrada/salida):

- Horas pico: 7, 8, 9, 17, 18, 19.


In [ ]:
peak_hours = [7, 8, 9, 17, 18, 19]
df["peak_hour"] = df["hr"].astype(int).isin(peak_hours).astype(int)
df[["hr", "peak_hour"]].head(10)

### 4.2. Conversión explícita de `dteday` a fecha (opcional)

La variable `dteday` está almacenada como texto. La convertimos a tipo fecha para facilitar análisis temporales (aunque no se usará directamente como predictor en el modelo lineal).


In [ ]:
if "dteday" in df.columns:
    df["dteday"] = pd.to_datetime(df["dteday"])
df[["dteday", "yr", "mnth", "weekday"]].head()

## 5. Creación de variables dummies

Para poder incluir variables categóricas en el modelo de regresión, creamos variables indicadoras (dummies) usando `pandas.get_dummies`, evitando la trampa de las variables ficticias con `drop_first=True`.


In [ ]:
X_raw = df[base_features + ["peak_hour"]].copy()

cat_features = [
    "hr", "weekday", "workingday", "season", "yr", "weathersit"
]
num_features = [col for col in X_raw.columns if col not in cat_features]

X_dummies = pd.get_dummies(
    X_raw,
    columns=cat_features,
    drop_first=True,
)

X_dummies.head()

Verificamos el tamaño del nuevo conjunto de predictores con dummies:


In [ ]:
X_dummies.shape

## 6. Construcción y guardado del dataset final para modelado

Definimos el vector objetivo `y` y la matriz de predictores `X`, y guardamos una versión preparada del dataset para reutilizarla en capítulos posteriores.


In [ ]:
y = df[target_col].copy()
X = X_dummies.copy()

X.head()

In [ ]:
# Unimos X e y en un solo DataFrame para inspección y posible guardado
df_model = X.copy()
df_model[target_col] = y
df_model.head()

In [ ]:
# Guardamos el dataset preparado (opcional pero útil para reutilizar)
output_path = "../data/hour_prepared.csv"
df_model.to_csv(output_path, index=False)
output_path

## 7. Resumen del capítulo

En este capítulo realizamos las siguientes tareas:

- Cargamos el dataset original `hour.csv` y verificamos estructura, tipos y duplicados.
- Convertimos variables categóricas al tipo `category`.
- Definimos un conjunto inicial de variables explicativas basado en clima y calendario.
- Creamos una característica adicional de hora pico (`peak_hour`).
- Generamos variables dummies para las variables categóricas relevantes.
- Construimos un dataset preparado (`df_model`) y lo guardamos en `../data/hour_prepared.csv` para usarlo en los capítulos de modelado.

En el próximo capítulo comenzaremos con el **modelo base de regresión lineal múltiple**, utilizando este conjunto de datos ya procesado.
